# Data Retrieval Google Trends

| Field | Details |
|---|---|
| **Time window** | 5 Jul 2024 – 4 Nov 2024 |
| **Source** | Google Trends — `pytrends` Python wrapper |
| **Method** | API — daily interest over time · daily interest by US state · related queries |
| **Keywords** | trump · kamala · vance · walz · election 2024 |
| **Volume** | Daily time series (5 keywords) · trump interest for 51 US states |
| **Saved to** | `Data/1_Bronze/Google Trends/` — one CSV per query type |

### trends_daily_stitched.csv 123 rows × 6 columns

| Column | Type | Description |
|---|---|---|
| `date` | date | Date (daily) |
| `trump`, `kamala`, `vance`, `walz`, `election 2024` | int | Search interest index (0–100) per keyword |

### trends_by_state.csv 51 rows × 6 columns

| Column | Type | Description |
|---|---|---|
| `geoName` | str | US state name |
| `trump`, `kamala`, `vance`, `walz`, `election 2024` | int | Relative search interest per state (0–100) |

### trump_daily_by_state.csv 123 rows × 52 columns

| Column | Type | Description |
|---|---|---|
| `date` | date | Date (daily) |
| `Alabama`, `Alaska`, … `Wyoming` | int | Daily Trump search interest index (0–100) per US state |

<!-- toc -->
## Contents
- [Setup](#setup)
- [1. Daily Interest Over Time](#1-daily-interest-over-time)
  - [trends_daily_stitched.csv columns](#trends_daily_stitchedcsv-columns)
- [2. Interest by Region](#2-interest-by-region)
  - [trends_by_state.csv columns](#trends_by_statecsv-columns)
- [3. Daily Interest by State](#3-daily-interest-by-state)
  - [trump_daily_by_state.csv columns](#trump_daily_by_statecsv-columns)


## Setup

In [3]:
import time
import pandas as pd
from pytrends.request import TrendReq

KEYWORDS = ['trump', 'kamala', 'vance', 'walz', 'election 2024']

GEO           = 'US'
CATEGORY      = 0
WINDOW_1      = ('2024-07-05', '2024-10-02')
WINDOW_2      = ('2024-08-26', '2024-11-04')
OVERLAP_START = '2024-08-26'
OVERLAP_END   = '2024-10-02'
ANCHOR        = 'trump'
# All 5 keywords fit in one pytrends batch (max = 5)
BATCHES = [KEYWORDS]
BRONZE_PATH = '../Data/1_Bronze/google_trends/'

pytrends = TrendReq(hl='en-US', tz=0, timeout=(10, 25))
tf1     = f"{WINDOW_1[0]} {WINDOW_1[1]}"
tf2     = f"{WINDOW_2[0]} {WINDOW_2[1]}"
full_tf = "2024-07-05 2024-11-04"

In [4]:
def fetch_window(timeframe, keywords, geo=GEO):
    """Fetch interest_over_time for a single window."""
    pytrends.build_payload(keywords, cat=CATEGORY, timeframe=timeframe, geo=geo)
    df = pytrends.interest_over_time()
    if df.empty:
        raise ValueError(f'Empty response for timeframe={timeframe}, geo={geo}')
    df = df.drop(columns=['isPartial'], errors='ignore')
    df.index = pd.to_datetime(df.index)
    return df

def stitch(df_w1, df_w2, keywords):
    """Rescale window 2 to window 1 using the overlap period, then concatenate."""
    w1_ov = df_w1.loc[OVERLAP_START:OVERLAP_END]
    w2_ov = df_w2.loc[OVERLAP_START:OVERLAP_END]
    df_w2s = df_w2.copy()
    for kw in keywords:
        m1, m2 = w1_ov[kw].mean(), w2_ov[kw].mean()
        df_w2s[kw] = (df_w2s[kw] * (m1 / m2 if m2 else 1.0)).clip(0, 100)
    w1_end = pd.Timestamp(WINDOW_1[1])
    out = pd.concat([df_w1, df_w2s.loc[w1_end + pd.Timedelta(days=1):]]).sort_index()
    return out[~out.index.duplicated(keep='first')]

## 1. Daily Interest Over Time

### trends_daily_stitched.csv columns

| Column | Type | Description |
|---|---|---|
| `date` | date | Date (daily) |
| `trump`, `kamala`, `vance`, `walz`, `election 2024` | int | Search interest index (0–100) per keyword |

In [5]:
# Fetch daily interest_over_time — single batch, just stitch the two windows
w1 = fetch_window(tf1, KEYWORDS)
time.sleep(10)
w2 = fetch_window(tf2, KEYWORDS)

df_combined = stitch(w1, w2, KEYWORDS)[KEYWORDS]
df_combined.index.name = 'date'
df_combined.to_csv(f'{BRONZE_PATH}trends_daily_stitched.csv')
print(f'trends_daily_stitched.csv: {df_combined.shape[0]} days x {df_combined.shape[1]} keywords')

trends_daily_stitched.csv: 123 days x 5 keywords


## 2. Interest by Region

### trends_by_state.csv columns

| Column | Type | Description |
|---|---|---|
| `geoName` | str | US state name |
| `trump`, `kamala`, `vance`, `walz`, `election 2024` | int | Relative search interest per state (0–100) |

In [6]:
# Fetch interest_by_region — single batch, no cross-batch normalisation needed
time.sleep(5)
pytrends.build_payload(KEYWORDS, cat=CATEGORY, timeframe=full_tf, geo=GEO)
df_by_region = pytrends.interest_by_region(resolution='REGION', inc_low_vol=True, inc_geo_code=False)
df_by_region = df_by_region[KEYWORDS]
df_by_region.to_csv(f'{BRONZE_PATH}trends_by_state.csv')
print(f'trends_by_state.csv: {df_by_region.shape[0]} states x {df_by_region.shape[1]} keywords')

trends_by_state.csv: 51 states x 5 keywords


## 3. Daily Interest by State

### trump_daily_by_state.csv columns

| Column | Type | Description |
|---|---|---|
| `date` | date | Date (daily) |
| `Alabama`, `Alaska`, … `Wyoming` | int | Daily Trump search interest index (0–100) per US state |

In [7]:
# Fetch daily trump interest for each of the 51 US states (stitched across two windows)
# Note: 51 states x 2 windows = 102 requests — allow ~20 min for rate-limit sleeps
STATE_CODES = {
    'Alabama': 'US-AL', 'Alaska': 'US-AK', 'Arizona': 'US-AZ',
    'Arkansas': 'US-AR', 'California': 'US-CA', 'Colorado': 'US-CO',
    'Connecticut': 'US-CT', 'Delaware': 'US-DE', 'District of Columbia': 'US-DC',
    'Florida': 'US-FL', 'Georgia': 'US-GA', 'Hawaii': 'US-HI',
    'Idaho': 'US-ID', 'Illinois': 'US-IL', 'Indiana': 'US-IN',
    'Iowa': 'US-IA', 'Kansas': 'US-KS', 'Kentucky': 'US-KY',
    'Louisiana': 'US-LA', 'Maine': 'US-ME', 'Maryland': 'US-MD',
    'Massachusetts': 'US-MA', 'Michigan': 'US-MI', 'Minnesota': 'US-MN',
    'Mississippi': 'US-MS', 'Missouri': 'US-MO', 'Montana': 'US-MT',
    'Nebraska': 'US-NE', 'Nevada': 'US-NV', 'New Hampshire': 'US-NH',
    'New Jersey': 'US-NJ', 'New Mexico': 'US-NM', 'New York': 'US-NY',
    'North Carolina': 'US-NC', 'North Dakota': 'US-ND', 'Ohio': 'US-OH',
    'Oklahoma': 'US-OK', 'Oregon': 'US-OR', 'Pennsylvania': 'US-PA',
    'Rhode Island': 'US-RI', 'South Carolina': 'US-SC', 'South Dakota': 'US-SD',
    'Tennessee': 'US-TN', 'Texas': 'US-TX', 'Utah': 'US-UT',
    'Vermont': 'US-VT', 'Virginia': 'US-VA', 'Washington': 'US-WA',
    'West Virginia': 'US-WV', 'Wisconsin': 'US-WI', 'Wyoming': 'US-WY',
}

state_series = {}
for i, (state, code) in enumerate(STATE_CODES.items()):
    print(f'[{i+1}/{len(STATE_CODES)}] {state}')
    try:
        w1 = fetch_window(tf1, ['trump'], geo=code)
        time.sleep(10)
        w2 = fetch_window(tf2, ['trump'], geo=code)
        time.sleep(10)
        state_series[state] = stitch(w1, w2, ['trump'])['trump']
    except Exception as e:
        print(f'  ERROR: {e} - skipping')
        time.sleep(15)

df_trump_states = pd.DataFrame(state_series)
df_trump_states.index.name = 'date'
df_trump_states.to_csv(f'{BRONZE_PATH}trump_daily_by_state.csv')
print(f'trump_daily_by_state.csv: {df_trump_states.shape[0]} days x {df_trump_states.shape[1]} states')

[1/51] Alabama
[2/51] Alaska
[3/51] Arizona
[4/51] Arkansas
[5/51] California
[6/51] Colorado
[7/51] Connecticut
[8/51] Delaware
[9/51] District of Columbia
[10/51] Florida
[11/51] Georgia
[12/51] Hawaii
[13/51] Idaho
[14/51] Illinois
[15/51] Indiana
[16/51] Iowa
[17/51] Kansas
[18/51] Kentucky
[19/51] Louisiana
  ERROR: HTTPSConnectionPool(host='trends.google.com', port=443): Read timed out. (read timeout=10) - skipping
[20/51] Maine
[21/51] Maryland
[22/51] Massachusetts
[23/51] Michigan
[24/51] Minnesota
[25/51] Mississippi
[26/51] Missouri
[27/51] Montana
  ERROR: The request failed: Google returned a response with code 429 - skipping
[28/51] Nebraska
[29/51] Nevada
[30/51] New Hampshire
[31/51] New Jersey
[32/51] New Mexico
[33/51] New York
[34/51] North Carolina
[35/51] North Dakota
[36/51] Ohio
[37/51] Oklahoma
[38/51] Oregon
[39/51] Pennsylvania
[40/51] Rhode Island
[41/51] South Carolina
[42/51] South Dakota
[43/51] Tennessee
[44/51] Texas
[45/51] Utah
[46/51] Vermont
[47/51] 

In [8]:
# Patch: re-fetch missing states (Louisiana, Montana) with retries
df_trump_states = pd.read_csv(f'{BRONZE_PATH}trump_daily_by_state.csv', index_col='date')
df_trump_states.index = pd.to_datetime(df_trump_states.index)

MISSING_STATES = {'Louisiana': 'US-LA', 'Montana': 'US-MT'}

for state, code in MISSING_STATES.items():
    print(f'Fetching {state}...')
    success = False
    for attempt in range(3):
        try:
            w1 = fetch_window(tf1, ['trump'], geo=code)
            time.sleep(15)
            w2 = fetch_window(tf2, ['trump'], geo=code)
            time.sleep(15)
            df_trump_states[state] = stitch(w1, w2, ['trump'])['trump']
            print(f'  OK')
            success = True
            break
        except Exception as e:
            print(f'  Attempt {attempt+1}/3 failed: {e}')
            time.sleep(60)
    if not success:
        print(f'  FAILED after 3 attempts — {state} still missing')

# Sort columns alphabetically and save
df_trump_states = df_trump_states.reindex(sorted(df_trump_states.columns), axis=1)
df_trump_states.index.name = 'date'
df_trump_states.to_csv(f'{BRONZE_PATH}trump_daily_by_state.csv')
print(f'trump_daily_by_state.csv updated: {df_trump_states.shape[0]} days x {df_trump_states.shape[1]} states')
print('Missing states:', [s for s in ['Louisiana', 'Montana'] if s not in df_trump_states.columns])


Fetching Louisiana...
  OK
Fetching Montana...
  OK
trump_daily_by_state.csv updated: 123 days x 51 states
Missing states: []
